# Inspection des donnees brutes

- **Objectif** : Avoir une idee sur les donnees avec lesquelles on va travailler, pour qu'on pouisse les transofrmer en un seul jeu de donnees avec une variable cible (Churn), et des variables explicatives (features).

In [ ]:
#Importation des libraries necessaires
import numpy as np
import pandas as pd
import seaborn as sb 
import matplotlib.pyplot as plt
from config import (
    RAW_INSCRIPTIONS,
    RAW_EVALUATIONS,
    RAW_ACTIVITES,
    RAW_PROGRAMMES,
    RAW_ETUDIANTS,
)

In [ ]:
# Importation des donnees

students = pd.read_csv(RAW_ETUDIANTS,dtype={"genre": "category"})
programs = pd.read_csv(RAW_PROGRAMMES,dtype={"filiere": "string"})
inscriptions = pd.read_csv(RAW_INSCRIPTIONS,parse_dates=["date_debut", "date_annulation"], dtype={"resultat_final": "category"})
evaluations = pd.read_csv(RAW_EVALUATIONS,parse_dates=["date_echeance", "date_soumission"], dtype={"type_evaluation": "category"})
activities = pd.read_csv(RAW_ACTIVITES,parse_dates=["horodatage"], dtype={"type_activite": "category"})

In [ ]:
tables = {
    
    "etudiant": students,
    "programmes":programs,
    "inscriptions":inscriptions,
    "evaluations":evaluations,
    "activites virtuelles": activities,
}


- Verification des ids pour chaque table

In [ ]:
#Primary keys verification
ids = {
    "students": (students, "id_etudiant"),
    "programs": (programs, "id_programme"),
    "inscriptions": (inscriptions, "id_inscription"),
    "evaluations": (evaluations, "id_evaluation"),
    "activities": (activities, "id_activite"),
}
summary = []
for name, (table, id_col) in ids.items():
    summary.append({
        "Table": name,
        "id Column": id_col,
        "Total": len(table),
        "Null": table[id_col].isna().sum(),
        "Duplicate": table[id_col].duplicated().sum(),
        "Unique": table[id_col].is_unique,
    })
summary_df = pd.DataFrame(summary)
summary_df


In [ ]:
# Verification del'unicite des programmes pour chaque etudiant dans la table inscriptions
print(inscriptions.duplicated(subset=['id_etudiant', 'id_programme']).sum())

- Verification des cles etrangers

In [ ]:
# inscriptions avec etudiants et programmes
print("id des etudiants sont toutes dans inscriptions: ")
print(inscriptions['id_etudiant'].isin(students['id_etudiant']).sum()==inscriptions['id_etudiant'].count())

print("id des programmes sont toutes dans inscriptions: ")
print(inscriptions['id_programme'].isin(programs['id_programme']).sum()==inscriptions['id_programme'].count())

# evaluations avec inscripion
print("id des inscriptions sont toutes dans evaluations: ")
print(evaluations['id_inscription'].isin(inscriptions['id_inscription']).sum()==evaluations['id_inscription'].count())
    
# activites virtuelles avec inscriptions
print("id des inscriptions sont toutes dans activites virtuelles: ")
print(activities['id_inscription'].isin(inscriptions['id_inscription']).sum()==activities['id_inscription'].count())


In [ ]:
# Inspection inverse, s'il ya des valeurs qui n'existent pas comme cle etrangers

print('Les ids qui existent dans la table etudiants mais non dans la table inscriptions')
print(len(set(students['id_etudiant'])-set(inscriptions['id_etudiant'])))

print('Les ids qui existent dans la table programmes mais non dans la table inscriptions')
print(len(set(programs['id_programme'])-set(inscriptions['id_programme'])))

print('Les ids qui existent dans la table inscriptions mais non dans la table evaluations')
print(len(set(inscriptions['id_inscription'])-set(evaluations['id_inscription'])))

print('Les ids qui existent dans la table inscriptions mais non dans la table activites virtuelles')
print(len(set(inscriptions['id_inscription'])-set(activities['id_inscription'])))

- Verification des valeurs manquantes

In [ ]:
missing =[]
for name,table in tables.items():
    print(f'Table: {name}')
    for col in table.columns:
        missing.append({
            "Column": col,
            "Missing Values": table[col].isna().sum()
        })
    missing_df = pd.DataFrame(missing)
    print(missing_df)
    missing=[]


In [ ]:
inscriptions[inscriptions['date_debut'].isna()]

In [ ]:
pd.crosstab(inscriptions['resultat_final'],inscriptions['date_annulation'].notna())

In [ ]:
pd.crosstab(evaluations['note'].isna(),evaluations['type_evaluation'])

In [ ]:
missing_eval = evaluations[evaluations['note'].isna()]
missing_eval = missing_eval.merge(
    inscriptions[
        ["id_inscription","resultat_final","date_debut","date_annulation"]
    ],
    on='id_inscription',
    how='left'
)
print(missing_eval["resultat_final"].value_counts(dropna='False'))

- Verification de la validite des valeurs pour chaque variable des tables

In [ ]:
#Pour les variables numeriques
for name, table in tables.items():
    print(f'La table {name} \n')
    print(table.describe())
    print("\n")

In [ ]:
#Pour les variables categoriques
cat = ['genre','type_evaluation','semestre','resultat_final','type_activite','niveau_scolaire','annee_scolaire']
for name,table in tables.items():
    print(f'La table {name}:')
    for col in cat:
        if col in table.columns:
            print(f'la variable {col} contient les valeurs : {table[col].unique()} \n')


In [ ]:
#Pour les variables temporelles
annees = programs['annee_scolaire'].str.split('/', expand=True)
programs['date_debut_programme'] = pd.to_datetime(
    np.where(
        programs['semestre'] == 'Semestre 1',
        annees[0] + '-09-01',
        annees[1] + '-01-01',
    )
)

inscriptions_dates = inscriptions.merge(
    programs[['id_programme', 'date_debut_programme']],
    on='id_programme',
    how='left',
)

#Verification temporelle sur inscriptions
err_annul_debut = inscriptions_dates[
    inscriptions_dates['date_annulation'].notna()
    & (inscriptions_dates['date_annulation'] < inscriptions_dates['date_debut'])
]
err_annul_prog = inscriptions_dates[
    inscriptions_dates['date_annulation'].notna()
    & (inscriptions_dates['date_annulation'] < inscriptions_dates['date_debut_programme'])
]
debut_avant_prog = inscriptions_dates[
    inscriptions_dates['date_debut'].notna()
    & (inscriptions_dates['date_debut'] < inscriptions_dates['date_debut_programme'])
]

print(f"Inscriptions avec date_annulation < date_debut : {len(err_annul_debut)}")
print(f"Inscriptions avec date_annulation < date_debut_programme (T_0) : {len(err_annul_prog)}")
print(f"Inscriptions avec date_debut < date_debut_programme (T_0) : {len(debut_avant_prog)}")




In [ ]:
#Verification temporelle sur evaluations
eval_dates = evaluations.merge(
    inscriptions_dates[['id_inscription', 'date_debut', 'date_debut_programme', 'date_annulation']],
    on='id_inscription',
    how='left',
)

print(f"Evaluations avec date_soumission < date_debut : {(eval_dates['date_soumission'] < eval_dates['date_debut']).sum()}")
print(f"Evaluations avec date_echeance < date_debut : {(eval_dates['date_echeance'] < eval_dates['date_debut']).sum()}")
print(f"Evaluations avec date_soumission < date_debut_programme (T_0) : {(eval_dates['date_soumission'] < eval_dates['date_debut_programme']).sum()}")
print(f"Evaluations avec date_echeance < date_debut_programme (T_0) : {(eval_dates['date_echeance'] < eval_dates['date_debut_programme']).sum()}")
print(f"Evaluations avec date_soumission > date_annulation : {(eval_dates['date_soumission'] > eval_dates['date_annulation']).sum()}")

In [ ]:
# Verification temporelle sur activites_virtuelles
act_dates = activities.merge(
    inscriptions_dates[['id_inscription', 'date_debut', 'date_debut_programme', 'date_annulation']],
    on='id_inscription',
    how='left',
)

print(f"Activites avec horodatage < date_debut : {(act_dates['horodatage'] < act_dates['date_debut']).sum()}")
print(f"Activites avec horodatage < date_debut_programme (T_0) : {(act_dates['horodatage'] < act_dates['date_debut_programme']).sum()}")
print(f"Activites avec horodatage > date_annulation : {(act_dates['horodatage'] > act_dates['date_annulation']).sum()}")

## Conclusion